# Debugging

I seem to have run into some problems (as of Easter weekend - ~05.04). The aims of this notebook are to get disaggregated data to investigate potential causes for the bad clusterings.

The ideas were:
1. Check the data cleaning step
2. Plot curves of metrics against parameter changes // **kinda done**
3. Disaggregating data by time (decadal, seasonal)
4. Mean profile of each cluster
5. Spatial resampling (potentially before the PCA step)

# Code

## Bonus Installs (Ignore)
One off pip installs etc

In [ ]:
%pip install gsw gsw-xarray

In [ ]:
%pip install pint pint-xarray

## Setup

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt

import gsw_xarray as gsw
import hvplot.xarray

from time import time

RANDOM_SEED = 5

In [ ]:
data_path = './data/ACOD_CTD_v1.0.nc'

data = xr.open_dataset(data_path)

## NOTE: according to the paper accompanying the dataset, the PPT will differ from PSU by '0.01-0.05' across the dataset; for our purposes this is fine, but still worth noting nonetheless
## All cruises pre-August 1995 were in PPT (which is around 14000 profiles so nearly half); repeating with cleaned data shows around half the data is from before 1995 too.
data['Merged_Salinity'] = data.Salinity_PPT_with_QC_applied.combine_first(data.Salinity_PSS_with_QC_applied)

data_reduced = data[['Temperature', 'Merged_Salinity']]
data_info = data[['BOTTOM_DEPTH', 'REGION', 'CRUISE_NUMBER', 'CRUISE_NAME']]

interpolated_data = data_reduced.interpolate_na(dim='PRESSURE')

In [ ]:
upper_data = interpolated_data.sel(PRESSURE=slice(0, 300))
upper_data

### Merging Salinity
Working (for now) on the assumption that PPT and PSS are approximately interchangeable

In [ ]:
def count_nans(da):
    return da.isnull().sum()

salinity_ppt = upper_data.Salinity_PPT_with_QC_applied
salinity_pss = upper_data.Salinity_PSS_with_QC_applied

salinity_pss.combine_first(salinity_ppt).isnull().sum()

### (1) Data Cleaning

In [ ]:
def clean_dataset(ds, salinity_var='Merged_Salinity'):
    '''
    takes xarray dataset, returns numpy array ready for clustering

    returns training data, xarray dataset (for time/coordinate purposes)
    '''

    interpolated_ds = ds.interpolate_na(dim='PRESSURE')

    temp = interpolated_ds.Temperature.data
    temp_nan_mask = ~np.any(np.isnan(temp), axis=0).astype(bool)
    
    sal = interpolated_ds[salinity_var]
    sal_nan_mask = ~np.any(np.isnan(sal), axis=0).astype(bool)

    nan_mask = temp_nan_mask & sal_nan_mask

    masked_data = ds.sel(PROFILE=nan_mask)
    
    masked_temp = temp[:,nan_mask].T
    masked_sal = sal[:,nan_mask].T

    full_data = np.concatenate((masked_temp, masked_sal), axis=1)

    return full_data, masked_data

In [ ]:
temp_array = upper_data.interpolate_na(dim='PRESSURE').Temperature.data
sal_array = upper_data.interpolate_na(dim='PRESSURE').Merged_Salinity

temp_nan_mask = ~np.sum(np.isnan(temp_array),axis=0).astype(bool)
sal_nan_mask = ~np.sum(np.isnan(sal_array),axis=0).astype(bool)

nan_mask = temp_nan_mask & sal_nan_mask

# originally 29717 values; ended up with 23604 after temp cut, 12018 after salinity cut as well
clean_temperature = temp_array[:,nan_mask].T
clean_salinity = sal_array[:,nan_mask].T

masked_data = upper_data.sel(PROFILE=nan_mask)

# first 51 will be temperature (0-50 dbar incl), next 51 will be salinity (0-50 dbar incl)
clean_data = np.concatenate((clean_temperature, clean_salinity), axis=1)

clean_data.shape

### (3) Disaggregating Data

#### Temporal Patterns

In [ ]:
## Temporal distribution of data
month_nums = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
plt.hist(upper_data.TIME.dt.month, bins=month_nums, width=0.8)
plt.xticks(month_nums)

In [ ]:
from collections import Counter

counts = Counter(data.TIME.dt.year.data)

years, counts = zip(*dict(counts).items())

fig, ax = plt.subplots()

ax.stackplot(years, counts)

ax.set_ylim(0)

ax.set_ylabel('Number of Profiles')
ax.set_xlabel('Year')

#### Disaggregating by MONTH
Investigating seasonality of profiles / when in the year they were collected

In [ ]:
## monthly binning
monthly_groups = upper_data.groupby(upper_data.TIME.dt.month)
monthly_groups = [tup[1] for tup in list(monthly_groups)]

In [ ]:
from sklearn.decomposition import PCA

fig, ax = plt.subplots(figsize=(10, 10))

for m_id in [7, 8]:#range(1, 11):

    month_data = monthly_groups[m_id]
    training_data, masked_ds = clean_dataset(month_data)

    n_samples, n_features = training_data.shape
    
    pca = PCA(n_components=8)
    pca.fit(training_data)
    
    ax.plot(range(1, pca.n_components_+1), pca.explained_variance_ratio_, marker='o', label=f'Month {m_id+1}', alpha=0.6)

ax.legend()
ax.grid(True)

In [ ]:
from sklearn.mixture import GaussianMixture

def component_scores(data, min_comps=1, max_comps=12):
    '''
    returns AIC and BIC scores in two separate lists
    '''

    aic_scores = []
    bic_scores = []
    
    num_components = list(range(min_comps, max_comps+1))

    for n_comp in num_components:
        GMM = GaussianMixture(n_components=n_comp)
        GMM.fit(data)

        aic_scores.append(GMM.aic(data)) 
        bic_scores.append(GMM.bic(data))

    return aic_scores, bic_scores

In [ ]:
from sklearn.mixture import GaussianMixture

month = 7
training_data, masked_ds = clean_dataset(monthly_groups[month])

pca_comps = 10
pca = PCA(n_components=pca_comps)
t_data = pca.fit_transform(training_data)

aic_scores = []
bic_scores = []

num_comps = list(range(1, 13))

for n_comp in num_comps:
    GMM = GaussianMixture(n_components=n_comp)
    GMM.fit(t_data)
    
    aic_scores.append(GMM.aic(t_data))
    bic_scores.append(GMM.bic(t_data))

fig, ax = plt.subplots()

ax.plot(num_comps, aic_scores, marker='o', label='AIC Score')
ax.set_ylabel('AIC Score')
ax.set_xlabel('Number of Components')

bic_ax = ax.twinx()
bic_ax.plot(num_comps, bic_scores, marker='o', label='BIC Score', color='pink')

bic_ax.set_ylabel('BIC Score')

fig.legend(loc=10)
fig.suptitle(f'Goodness of Component number // PCA: {pca_comps}, Month: {month}')

#### Disaggregating by DECADE
Investigating how profiles vary by year

In [ ]:
## decadal binning
bins = [1970, 1980, 1990, 2000, 2010, 2022]

decadal_bins = upper_data.groupby_bins(upper_data.TIME.dt.year, bins=bins)
decadal_groups = [tup[1] for tup in list(decadal_bins)]

test_ds = decadal_groups[0]

In [ ]:
fig, axs = plt.subplots(3, 1)
axs = axs.flatten()

axs[0].hist(upper_data.TIME.dt.year, bins=47)
axs[1].hist(upper_data.TIME.dt.year, bins=50)
axs[2].hist(upper_data.TIME.dt.year, bins=bins)

In [ ]:
from sklearn.decomposition import PCA

fig, ax = plt.subplots(figsize=(8, 4))

for d_id, test_ds in enumerate(decadal_groups):

    pca = PCA(n_components=8)
    
    try:
        training_data, masked_ds = clean_dataset(test_ds, salinity_var='Salinity_PPT_with_QC_applied')
        pca.fit(training_data)

    except:
        training_data, masked_ds = clean_dataset(test_ds, salinity_var='Salinity_PSS_with_QC_applied')
        pca.fit(training_data)

    
    ax.plot(range(1, pca.n_components_+1), pca.explained_variance_ratio_, marker='o', alpha=0.6, label=f'Decade {d_id}')

ax.legend()
ax.grid(True)

In [ ]:
from sklearn.mixture import GaussianMixture

pca_final = PCA(n_components=4)

training_data, training_ds = clean_dataset(decadal_groups[0], salinity_var='Salinity_PPT_with_QC_applied')

reduced_data = pca_final.fit_transform(training_data)

aic = []
bic = []

n_comp = range(1, 11)

for comps in n_comp:
    GMM = GaussianMixture(n_components=comps)
    GMM.fit(reduced_data)

    aic.append(GMM.aic(reduced_data))
    bic.append(GMM.bic(reduced_data))

fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(n_comp, aic, label='AIC')
ax.plot(n_comp, bic, label='BIC')

In [ ]:
final_GMM = GaussianMixture(n_components=4)

gmm_clusters = final_GMM.fit_predict(reduced_data)

#### Data by DEPTH
Investigating the depth distribution of the profiles

We observe a massive chunk below 500m (actually mostly below 300m) -- this seems like a sensible point to cut off profiles. There is also a fairly large spike of profiles with a maximum depth of 1500m. I would hypothesise this is an instrument thing, where 'normal' CTD sensors only go down to 1500m, while the equivalent of deep ARGO CTD measurements go down to 2500m. This 'change in instrument' theory is potentially somewhat supported by the graph showing the PRESSURE column in the dataset is not linear, with a change in step size after 1500m.

In [ ]:
## define a profile depth by maximum depth [pressure level] - number of nans
## since has been interpolated, nans should start from the sea floor

def profile_depth(da):
    total_depth = len(da.PRESSURE)
    
    return total_depth + ~da.isnull().sum(dim='PRESSURE')

In [ ]:
depths = profile_depth(interpolated_data.Temperature)

fig, ax = plt.subplots()

ax.hist(depths.data, 80, cumulative=False, histtype='bar')

ax.set_ylabel('Frequency')
ax.set_xlabel('Number of values per profile')

ax.vlines(150, 0, 6000, color='black', label='150m', linestyle=':')
ax.vlines(300, 0, 6000, color='black', label='300m', linestyle=':')

ax.legend()

ax.set_title('Distribution of maximum profile depth (in Temperature)')

In [ ]:
fig, ax = plt.subplots()

ax.plot(data.PRESSURE)

ax.set_xlabel('n')
ax.set_ylabel('Pressure')

ax.set_title('Pressure')

## Clustering

### PCA Component Selection

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=8)
pca.fit(clean_data)

plt.stem(range(1, pca.n_components_+1), pca.explained_variance_ratio_)
plt.grid(True)

plt.title('Explained Variance Ratios')

In [ ]:
## Dimensionality Reduction (PCA)
from sklearn.decomposition import PCA

pca = PCA(n_components=4)

reduced_data = pca.fit_transform(clean_data)
reduced_data.shape

### (2) GMM METRICS

In [ ]:
## Calculating Metrics for varying numbers of components
start_time = time()

from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score as SS, davies_bouldin_score as DBI, calinski_harabasz_score as CHI

component_nums = list(range(2, 12))

sil_scores = []
db_scores = []
ch_scores = []

for n in component_nums:
    # model = KMeans(n_clusters=n, random_state=RANDOM_SEED)
    # model = GaussianMixture(n_components=n, random_state=RANDOM_SEED)
    model = AgglomerativeClustering(n_clusters=n, linkage='ward')

    clusters = model.fit_predict(reduced_data)

    sil_scores.append(SS(reduced_data, clusters))
    db_scores.append(DBI(reduced_data, clusters))
    ch_scores.append(CHI(reduced_data, clusters))

print(f'Done in {time() - start_time:.02f}s')

The **silhouette score** varies between -1 and 1, with 1 being the best and -1 being the worst score possible. 1 implies perfect clustering, 0 implies overlapping clusters and -1 implies the wrong classification. We thus want to __maximise this score__.

The **Davies Bouldin Score** has a minumum value of 0, with lower values being better. It is a measure of the 'average similarity' of each cluster to its most similar cluster.  *I assume this means that it penalises overlapping clusters specifically, so we would expect to extract a maximum number of clusters from this metric*.

The **Calinski Harabasz Score** is the ratio between the sum of between-cluster dispersion and within-cluster dispersion (sklearn). A higher value is generally better as this indicates the points are generally more spread out between clusters than they are within.

The TL;DR is we are looking for the metrics to be: **MAX, MIN, MAX**

In [ ]:
fig, axs = plt.subplots(3, 1, sharex=True, figsize=(10, 15), subplot_kw={'xlabel': 'Number of Components', 'ylabel': 'Score'})
axs = axs.flatten()

axs[0].plot(component_nums, sil_scores, color='blue', marker='o', alpha=0.8, label='Silhouette Score')

axs[1].plot(component_nums, db_scores, color='gray', marker='o', alpha=0.8, label='Davies Bouldin Score')

axs[2].plot(component_nums, ch_scores, color='orange', marker='o', alpha=0.8, label='Calinski Harabasz Score')

axs[0].set_xticks(component_nums)

fig.legend()
plt.show()

In [ ]:
## Clustering (GMM) -- final clustering with choice based on above

from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=4, random_state=RANDOM_SEED)

gmm_clusters = gmm.fit_predict(reduced_data)
gmm_probs = gmm.predict_proba(reduced_data)

### Alternative Clustering Methods

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=6, random_state=RANDOM_SEED)

gmm_clusters = kmeans.fit_predict(reduced_data)

In [ ]:
from sklearn.cluster import AgglomerativeClustering

agg_clustering = AgglomerativeClustering(n_clusters=5, linkage='ward')

gmm_clusters = agg_clustering.fit_predict(reduced_data)

## Analysis

### Extremely Specific Plots

#### 1. Dendrogram
Code is copied directly from the [sklearn docs](https://scikit-learn.org/stable/auto_examples/cluster/plot_agglomerative_dendrogram.html#sphx-glr-auto-examples-cluster-plot-agglomerative-dendrogram-py)

In [ ]:
from scipy.cluster.hierarchy import dendrogram

def plot_dendrogram(model, **kwargs):
    # Create linkage matrix and then plot the dendrogram

    # create the counts of samples under each node
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1  # leaf node
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count

    linkage_matrix = np.column_stack(
        [model.children_, model.distances_, counts]
    ).astype(float)

    # Plot the corresponding dendrogram
    dendrogram(linkage_matrix, **kwargs)

In [ ]:
from sklearn.cluster import AgglomerativeClustering

# setting distance_threshold=0 ensures we compute the full tree.
model = AgglomerativeClustering(distance_threshold=0, n_clusters=None, linkage='ward')

fig, ax = plt.subplots(figsize=(20, 10))

model = model.fit(reduced_data)
ax.set_title("Hierarchical Clustering Dendrogram")
# plot the top three levels of the dendrogram
plot_dendrogram(model, truncate_mode="lastp", p=20, ax=ax)
ax.set_xlabel("Number of points in node (or index of point if no parenthesis).")
plt.show()

### Geographical Plots

#### Setup

In [ ]:
## Color mapping functions

def dec_to_hex(n):
    '''
    for use in rgba_to_hex

    scales numbers from 0 to 1 -> hex from 1 -> 225
    '''
    
    decimal_value = round(n * 256)
    if decimal_value == 256:
        decimal_value = 255
    hex_val = f'{decimal_value:02x}'

    return hex_val

def rgba_to_hex(arr):
    '''
    assuming input is in form from colormap
    (r, g, b, a)
    '''

    rgb_values = arr[:3]

    return '#' + ''.join([dec_to_hex(num) for num in rgb_values])

#### Plots on Maps (1)
Plotting each cluster individually

In [ ]:
cmap = mpl.colormaps['viridis']
_cid = np.linspace(0, 1, len(set(gmm_clusters)))

colors = [
    rgba_to_hex(cmap(i)) for i in _cid
]

pt_clrs = list(map(lambda i: colors[i], gmm_clusters))

colors

In [ ]:
## plot masked data

cluster_num = len(set(gmm_clusters))

import cartopy.crs as ccrs
import cartopy.feature as cfeature

full_lats = masked_data.LATITUDE
full_lons = masked_data.LONGITUDE

fig, axs = plt.subplots(cluster_num, 1, figsize=(8,4*cluster_num), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)})
axs = axs.flatten()

for cluster_id in range(cluster_num):
    ax = axs[cluster_id]
    cluster_mask = gmm_clusters == cluster_id
    
    cluster_lats = full_lats[cluster_mask]
    cluster_lons = full_lons[cluster_mask]
    
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS)
    ax.add_feature(cfeature.LAND)
    
    ax.scatter(full_lons+180, full_lats, s=1, alpha=0.4, c='Black')
    ax.scatter(cluster_lons+180, cluster_lats, s=2, alpha=0.7, c='Red')

    ax.set_title(f'Cluster {cluster_id}')

#### Plots on Maps (2) --- GMM SPECIFIC
Plotting the probability each point is in each cluster

In [ ]:
probabilities = GMM_probs

### Profiles
Inspecting properties of profiles post-clustering (Mostly Temperature and Salinity Plots)

#### Single Profile (backup)

In [ ]:
cluster_mask = gmm_clusters == 2

cluster_ds = masked_data.sel(PROFILE=cluster_mask)

PRESSURE = cluster_ds.PRESSURE
DEPTHS = -1 * PRESSURE

cluster_temp = cluster_ds.Temperature
cluster_salinity_PPT = cluster_ds.Salinity_PPT_with_QC_applied
cluster_salinity_PSS = cluster_ds.Salinity_PSS_with_QC_applied

temp_stats = average_profile(cluster_temp)
sal_stats = average_profile(cluster_salinity_PSS)

fig, axs = plt.subplots(1, 3, figsize=(20, 6))
axs = axs.flatten()

axs[0].set_title('Temperature')
axs[1].set_title('Salinity (PPT)')

plot_average_profile(temp_stats, axs[0])
plot_average_profile(sal_stats, axs[1])

ax = axs[2]

ax.plot(sal_stats[0], temp_stats[0])
ax.set_title('Average T-S Profile')
ax.set_ylabel('Temperature (C)')
ax.set_xlabel('Salinity (PPT)')

In [ ]:
from collections import Counter

cluster_mask = gmm_clusters == 0
cluster_ds = masked_data.sel(PROFILE=cluster_mask)

cluster_times = cluster_ds.TIME
labels, heights = np.array(Counter(cluster_times.dt.year.data).most_common()).T

plt.bar(labels, heights)

#### Mean Cluster Profile

In [ ]:
def average_profile(ds):
    '''
    get mean and std from ds
    '''

    return (ds.mean(dim='PROFILE'), ds.std(dim='PROFILE'))

def plot_average_profile(stats, ax, plot_color='blue', fill_color='lightgray', label='', DEPTHS=None):
    '''
    plots <var>-depth plot
    '''

    assert np.all(stats[0].PRESSURE == stats[1].PRESSURE), 'Pressure values in mean/std do not match :('

    if DEPTHS == None:
        PRESSURE = stats[0].PRESSURE
        DEPTHS = -1 * PRESSURE

    ax.plot(stats[0], DEPTHS, label=label, color=plot_color)
    ax.fill_betweenx(DEPTHS, stats[0]-stats[1], stats[0]+stats[1], alpha=0.6, color=fill_color)

    return ax

In [ ]:
total_clusters = len(set(gmm_clusters))

PRESSURE = masked_data.PRESSURE
DEPTHS = -1 * PRESSURE

cluster_means = []

ylims = (DEPTHS.min(), DEPTHS.max())

for cluster_id in range(total_clusters):

    cluster_mask = gmm_clusters == cluster_id
    cluster_ds = masked_data.sel(PROFILE=cluster_mask)
    
    cluster_temp = cluster_ds.Temperature
    cluster_salinity = cluster_ds.Merged_Salinity
    
    cluster_AS = gsw.SA_from_SP(SP=cluster_salinity, p=cluster_ds.PRESSURE, lon=cluster_ds.LONGITUDE, lat=cluster_ds.LATITUDE)
    cluster_CT = gsw.conversions.pt0_from_t(SA=cluster_AS, t=cluster_ds.Temperature, p=cluster_ds.PRESSURE)
    
    # temp_stats = average_profile(cluster_temp)
    # sal_stats = average_profile(cluster_salinity)

    temp_stats = average_profile(cluster_CT)
    sal_stats = average_profile(cluster_AS)

    cluster_means.append((temp_stats, sal_stats))
    
    fig, axs = plt.subplots(1, 3, figsize=(20, 6))
    axs = axs.flatten()

    axs[0].set_ylim(ylims[0], ylims[1])
    axs[0].set_title('Temperature')

    axs[1].set_ylim(ylims[0], ylims[1])
    axs[1].set_title('Salinity (PPT)')
    
    plot_average_profile(temp_stats, axs[0])
    plot_average_profile(sal_stats, axs[1])
    
    ax = axs[2]
    
    ax.plot(sal_stats[0], temp_stats[0], marker='o')
    ax.set_title('Average T-S Profile')
    ax.set_ylabel('Temperature (C)')
    ax.set_xlabel('Salinity (PSS)')

    fig.suptitle(f'Cluster {cluster_id}')

In [ ]:
from datetime import datetime

## mean plot

fig, axs = plt.subplots(1, 2, figsize=(20, 10), sharey=True, subplot_kw={'ylim': ylims})

for c_id, cluster in enumerate(cluster_means):
    plot_average_profile(cluster[0], axs[0], label=c_id, plot_color=colors[c_id])
    plot_average_profile(cluster[1], axs[1], label=c_id, plot_color=colors[c_id])

axs[0].legend()

axs[0].set_xlabel(r'In-Situ Temperature ($^\circ$C)')
axs[0].set_title('Temperature')

axs[1].set_ylabel('')
axs[1].set_title('Salinity')

title = 'GMM Clusters'

fig.suptitle(title, size=24)

date = datetime.today().strftime('%d.%m')
filetype = 'png'
fname = f'{date}-{title}.{filetype}'

# plt.savefig(f'./PLOTS/{fname}', format=filetype, transparent=False)

**What do these clusters show?**

*quick note on colors* <br>
The colors are ever-changing -- refer to the cluster numbers instead (given the seed has been set, this should be consistent when there are 4 K-means clusters.

*GMM* <br>
We once again see the distinction of summer and winter waters, but what is particularly interesting is in the salinity plots -- we observe some variation in the ACC waters, but the Bering Sea waters (0, 1) are near enough constant all the way down, which is likely due to the (well-documented) permanent halocline in the Bering Sea.

*K-Means* <br>
We can see two clear bodies of water from the deep-water profiles -- clearly it seems that 0, 1 (blue and purple; hereby labelled water mass A) are separate from 2, 3 (yellow and green; hereon water mass B) from their deep-water temperature profiles, but we see the pairs of profiles seem to diverge at the upper layers, starting from around 170m in A and a little deeper at just over 200m in B, with one clear warmer and one clear colder profile in each case. In particular, we notice the the colder profile seems quite homogeneous (temperature-wise) down to around 300m, varying less than 2 degrees in B and less than 1 in A. These are clearly indicative of a summer and winter water being picked out (which can be verified from the plots below).

In [ ]:
## Potential Temperature / AS plots

n_pts = 50

AS_range = np.linspace(27, 36, n_pts)
PT_range = np.linspace(-2, 12, n_pts)

AS_grid = AS_range.reshape(-1, 1).repeat(n_pts, axis=1).T
PT_grid = PT_range.reshape(-1, 1).repeat(n_pts, axis=1)

sigma0 = gsw.density.sigma0(AS_grid, PT_grid)+1000

clevels = np.arange(np.floor(sigma0.min()), np.ceil(sigma0.max())+1, 1)

fig, ax = plt.subplots(figsize=(15, 10))

CONTOURS = ax.contour(AS_range, PT_range, sigma0, colors='lightgray', levels=clevels, alpha=0.4)
ax.clabel(CONTOURS, CONTOURS.levels)

for c_id, cmean in enumerate(cluster_means):

    pt_mean = cmean[0][0].data
    sa_mean = cmean[1][0].data

    ax.scatter(sa_mean, pt_mean, label=f'Cluster {c_id}', marker='o', s=5, alpha=1)

ax.set_xlabel('Absolute Salinity')
ax.set_ylabel(r'Potential Temperature')
ax.legend()



#plt.savefig('./PLOTS/TS plot.png', format='png')

In [ ]:
np.arange(np.floor(1.5), 10, 1)

In [ ]:
## Plot of all profiles

## setup same as above
n_pts = 80

pt_size = 10

full_AS = gsw.SA_from_SP(SP=masked_data.Merged_Salinity, p=masked_data.PRESSURE, lon=masked_data.LONGITUDE, lat=masked_data.LATITUDE)
full_PT = gsw.conversions.pt0_from_t(SA=full_AS, t=masked_data.Temperature, p=masked_data.PRESSURE)

AS_min = min(float(full_AS.min().data), 27)
AS_max = max(float(full_AS.max().data), 36)
AS_range = np.linspace(AS_min, AS_max, n_pts)

PT_min = min(float(full_PT.min().data), -2)
PT_max = max(float(full_PT.max().data)+(pt_size/25), 12)
PT_range = np.linspace(PT_min, PT_max, n_pts)

xticks = np.arange(np.floor(AS_min), np.ceil(AS_max)+1, 1)
yticks = np.arange(np.floor(PT_min), np.ceil(PT_max)+1, 1)

AS_grid = AS_range.reshape(-1, 1).repeat(n_pts, axis=1).T
PT_grid = PT_range.reshape(-1, 1).repeat(n_pts, axis=1)

sigma0 = gsw.density.sigma0(AS_grid, PT_grid)+1000

clevels = np.arange(np.floor(sigma0.min()), np.ceil(sigma0.max())+1, 1)

fig, ax = plt.subplots(figsize=(15, 10))

ax.set_xticks(xticks)
ax.set_yticks(yticks)

CONTOURS = ax.contour(AS_range, PT_range, sigma0, colors='lightgray', levels=clevels, alpha=0.7, zorder=-1)
ax.clabel(CONTOURS, CONTOURS.levels)

## 

total_clusters = len(set(gmm_clusters))

PRESSURE = masked_data.PRESSURE
DEPTHS = -1 * PRESSURE

for cluster_id in range(total_clusters):

    cluster_mask = gmm_clusters == cluster_id
    cluster_ds = masked_data.sel(PROFILE=cluster_mask)
    
    cluster_temp = cluster_ds.Temperature
    cluster_salinity = cluster_ds.Merged_Salinity
    
    cluster_ds['Absolute_Salinity'] = gsw.SA_from_SP(SP=cluster_salinity, p=cluster_ds.PRESSURE, lon=cluster_ds.LONGITUDE, lat=cluster_ds.LATITUDE)
    cluster_ds['Potential_Temperature'] = gsw.conversions.pt0_from_t(SA=cluster_ds.Absolute_Salinity, t=cluster_ds.Temperature, p=cluster_ds.PRESSURE)
    
    ax.scatter(cluster_ds.Absolute_Salinity.data.ravel(), cluster_ds.Potential_Temperature.data.ravel(), s=pt_size, color=colors[cluster_id], alpha=0.2, label=f'Cluster {cluster_id}')

ax.set_ylabel(r'Potential Temperature ($^\circ C$)')
ax.set_xlabel('Absolute Salinity')
ax.set_title('KNN Clusters')
ax.legend()

#plt.savefig('./PLOTS/All_data_GMM.png', format='png')

#### Temporal Distribution

In [ ]:
from collections import Counter

total_clusters = len(set(gmm_clusters))
month_nums = list(range(1, 13))

for cluster_id in range(total_clusters):

    cluster_mask = gmm_clusters == cluster_id
    cluster_ds = masked_data.sel(PROFILE=cluster_mask)

    cluster_times = cluster_ds.TIME

    fig, axs = plt.subplots(1, 2, figsize=(14, 6))
    axs = axs.flatten()

    cluster_times = cluster_ds.TIME
    
    ## YEAR DISTRIBUTION
    labels, heights = np.array(Counter(cluster_times.dt.year.data).most_common()).T
    axs[0].bar(labels, heights)

    ## MONTHLY DISTRIBUTION
    labels, heights = np.array(Counter(cluster_times.dt.month.data).most_common()).T
    axs[1].bar(labels, heights)
    axs[1].set_xticks(month_nums)

    fig.suptitle(f'Cluster {cluster_id}')

### Density
Using gsw to calculate the in-situ density

In [ ]:
## we add all the variables to this one
ref_ds = masked_data.copy()

## absolute from practical salinity
# takes Practical Salinity, pressure (depth), longitude, latitude
ref_ds['Absolute_Salinity'] = gsw.SA_from_SP(SP=ref_ds.Merged_Salinity, p=ref_ds.PRESSURE, lon=ref_ds.LONGITUDE, lat=ref_ds.LATITUDE)
ref_ds['Conservative_Temperature'] = gsw.conversions.CT_from_t(SA=ref_ds.Absolute_Salinity, t=ref_ds.Temperature, p=ref_ds.PRESSURE)

## note this one is sigma_T
ref_ds['Density_Anomaly'] = gsw.density.sigma0(SA=ref_ds.Absolute_Salinity, CT=ref_ds.Conservative_Temperature)

ref_ds['is_Density'] = gsw.rho_t_exact(SA=ref_ds.Absolute_Salinity, t=ref_ds.Temperature, p=ref_ds.PRESSURE)

In [ ]:
ds_mean = average_profile(ref_ds.Density_Anomaly)

fig, ax = plt.subplots()

plot_average_profile(ds_mean, ax=ax)

ax.set_xlabel(r'Density Anomaly ($\sigma_T$)')

# Data Vis
For looking at bathymetry n stuff

And also for final plots

**Notes on final plots**

For Temperature-Salinity plots, obviously I first need to convert to Potential Temperature and Absolute Salinity, but then it would also be nice to have potential density contours along the back

I also need to plot the PCA maps to get an idea of what the GMM is seeing (if it is overly prioritising a singular PC, in which case some degree of normalisation is likely required)

Also potentially possible to use t-SNE for visualisation of data and cluster

In [ ]:
import xarray as xr
import hvplot.xarray
import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt

In [ ]:
ds = xr.open_dataset('http://iridl.ldeo.columbia.edu/SOURCES/.NOAA/.NGDC/.ETOPO5/dods')
ds = ds.rename({'X': 'lon', 'Y': 'lat'})
ds['elev_ocean'] = ds.elev.where(ds.elev<0).load()
ds

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
ds.elev_ocean.plot.pcolormesh(ax=ax)

In [ ]:
lat_range = (40, 90)
lon_range = (150, 250)

reduced_ds = ds.sel(lat=slice(lat_range[1], lat_range[0]), lon=slice(lon_range[0], lon_range[1]))
reduced_ds['sub_300'] = reduced_ds.elev_ocean.where(ds.elev<-300)
reduced_ds['shallow'] = reduced_ds.elev_ocean.where(ds.elev>-300)

reduced_ds

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(20, 20))

reduced_ds.elev_ocean.plot.pcolormesh(ax=axs[0])
reduced_ds.shallow.plot.pcolormesh(ax=axs[1])